In [3]:
import sys
import os
import pandas as pd
import importlib

# --- Path setup ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# --- Load framework ---
import ab_test_framework
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Load cleaned data ---
cleaned_path = os.path.join(project_root, 'data', 'cookie_cats_cleaned.csv')
df = pd.read_csv(cleaned_path)

print("Data shape:", df.shape)
print("Groups:", df['version'].value_counts().to_dict())

ab_test_framework.py loaded successfully.
ab_test_framework.py loaded successfully.
Data shape: (90188, 5)
Groups: {'gate_40': 45489, 'gate_30': 44699}


In [5]:
# --- Run framework on all three outcomes ---
results = {}

for outcome in ['sum_gamerounds', 'retention_1', 'retention_7']:
    results[outcome] = run_ab_test(
        data        = df,
        group_col   = 'version',
        outcome_col = outcome,
        group_a     = 'gate_30',
        group_b     = 'gate_40'
    )
    print(f"✓ {outcome} done — test: {results[outcome]['test_used']}, p={results[outcome]['p_value']}")

print("\nAll three outcomes analyzed.")

✓ sum_gamerounds done — test: mann_whitney_u, p=0.050892
✓ retention_1 done — test: chi_square, p=0.07501
✓ retention_7 done — test: chi_square, p=0.001639

All three outcomes analyzed.


In [6]:
# --- Build summary table ---
rows = []

for outcome, result in results.items():
    rows.append({
        'Outcome'                : outcome,
        'Test Used'              : result['test_used'],
        'Statistic'              : result['statistic'],
        'p-value'                : result['p_value'],
        'Significant?'           : 'Yes' if result['significant'] else 'No',
        'Effect Size'            : result['effect_size'],
        'Effect Type'            : result['effect_size_type'],
        'Effect Magnitude'       : result['effect_magnitude'],
        'Practically Meaningful?': 'No' if result['effect_magnitude'] == 'negligible' else 'Yes'
    })

summary_table = pd.DataFrame(rows)

print("=== A/B Test Results Summary Table ===\n")
print(summary_table.to_string(index=False))

# --- Save to CSV ---
summary_path = os.path.join(project_root, 'data', 'phase4_results_summary.csv')
summary_table.to_csv(summary_path, index=False)
print(f"\nSummary table saved to: {summary_path}")

=== A/B Test Results Summary Table ===

       Outcome      Test Used    Statistic  p-value Significant?  Effect Size   Effect Type Effect Magnitude Practically Meaningful?
sum_gamerounds mann_whitney_u 1.024286e+09 0.050892           No    -0.007504 rank_biserial       negligible                      No
   retention_1     chi_square 3.169800e+00 0.075010           No     0.005928     cramers_v       negligible                      No
   retention_7     chi_square 9.915300e+00 0.001639          Yes     0.010485     cramers_v       negligible                      No

Summary table saved to: C:\Users\R D\AB_Test_FrameWork\data\phase4_results_summary.csv


In [8]:
interpretation = """
╔══════════════════════════════════════════════════════════════════════╗
║         PHASE 4 — INTERPRETATION & BUSINESS RECOMMENDATION           ║
║         Cookie Cats A/B Test — Gate Position Experiment              ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXPERIMENT CONTEXT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Control   : gate_30 — gate appears at level 30 (n = 44,699)
  Treatment : gate_40 — gate appears at level 40 (n = 45,489)
  Question  : Does moving the gate from level 30 to level 40
              improve player engagement and retention?
  Design    : Properly randomized experiment — causal claims
              are valid (no confounding caveat needed)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOME 1: sum_gamerounds (Player Engagement)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Statistical result : NOT significant (p=0.0509, just above alpha=0.05)
  Effect size        : Rank-biserial r = -0.0075 (negligible)
  gate_30 median     : 17 rounds
  gate_40 median     : 16 rounds

  Plain language:
  Moving the gate to level 40 produced no statistically significant
  change in how many rounds players played. The p-value of 0.051 just
  barely missed the significance threshold — but even if it had crossed
  it, an effect size of -0.0075 is essentially zero. A difference of
  1 median round (17 vs 16) is not a meaningful business signal.

  Statistical vs Practical:
  Not significant AND not practically meaningful. This outcome gives
  no reason to prefer either gate position — engagement looks the
  same regardless of where the gate sits.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOME 2: retention_1 (1-Day Retention)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Statistical result : NOT significant (p=0.0750, above alpha=0.05)
  Effect size        : Cramer's V = 0.0059 (negligible)
  gate_30 rate       : 44.82%
  gate_40 rate       : 44.23%
  Difference         : 0.59 percentage points

  Plain language:
  There is no statistically significant difference in 1-day retention
  between the two gate positions. The 0.59 percentage point gap in
  favor of gate_30 is well within the range of random chance —
  p=0.075 does not meet the 0.05 threshold. We cannot conclude the
  gate position affects whether players return the next day.

  Statistical vs Practical:
  Not significant AND not practically meaningful. A 0.59 percentage
  point difference in retention would be operationally invisible —
  it would not move any business metric in a detectable way.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOME 3: retention_7 (7-Day Retention) — THE KEY FINDING
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Statistical result : SIGNIFICANT (p=0.0016, well below alpha=0.05)
  Effect size        : Cramer's V = 0.0105 (negligible)
  gate_30 rate       : 19.02%
  gate_40 rate       : 18.20%
  Difference         : 0.82 percentage points
  95% CI for diff    : (0.003103, 0.013263)

  Plain language:
  7-day retention is statistically significantly higher for gate_30
  than gate_40 — this result is real and not due to chance (p=0.0016).
  The confidence interval for the difference (0.31pp to 1.33pp) does
  not include zero, confirming the direction of the effect is reliable.
  gate_30 genuinely produces better 7-day retention than gate_40.

  Statistical vs Practical:
  THIS IS THE CRITICAL DISTINCTION. The result is statistically
  significant — but Cramer's V of 0.0105 is negligible by any
  standard benchmark. The absolute difference is 0.82 percentage
  points. At 90,000 players this becomes detectable, but in practice
  0.82pp retention difference does not represent a transformative
  business improvement.

  However — 7-day retention is the most important metric here.
  Even a small, consistent advantage in long-term retention compounds
  meaningfully at scale. If Cookie Cats has millions of installs,
  0.82pp more players retained at day 7 across millions of users
  is not trivially small in absolute player count terms.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESULTS SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Outcome          Significant?   Practically Meaningful?   Winner
  ───────────────────────────────────────────────────────────────
  sum_gamerounds   No             No                        Neither
  retention_1      No             No                        Neither
  retention_7      YES            Marginal at scale         gate_30
  ───────────────────────────────────────────────────────────────
  Overall                                                   gate_30
"""

print(interpretation)

# --- Save ---
interp_path = os.path.join(project_root, 'data', 'phase4_interpretation.txt')
with open(interp_path, 'w', encoding='utf-8') as f:
    f.write(interpretation)
print(f"Interpretation saved to: {interp_path}")


╔══════════════════════════════════════════════════════════════════════╗
║         PHASE 4 — INTERPRETATION & BUSINESS RECOMMENDATION           ║
║         Cookie Cats A/B Test — Gate Position Experiment              ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EXPERIMENT CONTEXT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Control   : gate_30 — gate appears at level 30 (n = 44,699)
  Treatment : gate_40 — gate appears at level 40 (n = 45,489)
  Question  : Does moving the gate from level 30 to level 40
              improve player engagement and retention?
  Design    : Properly randomized experiment — causal claims
              are valid (no confounding caveat needed)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTCOME 1: sum_gamerounds (Player Engagement)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  St

In [9]:
recommendation = """
╔══════════════════════════════════════════════════════════════════════╗
║              FINAL BUSINESS RECOMMENDATION & LIMITATIONS             ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL RECOMMENDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Do NOT move the gate to level 40.

  Keep the gate at level 30 (gate_30).

  Full recommendation statement:
  "Based on a properly randomized experiment of 90,188 players,
  moving the gate from level 30 to level 40 shows no improvement
  in player engagement or 1-day retention, and produces a small
  but statistically significant reduction in 7-day retention
  (19.02% vs 18.20%, p=0.0016). While the effect is modest in
  percentage-point terms, 7-day retention is the most meaningful
  long-term health metric available in this dataset, and gate_30
  outperforms gate_40 on every measured outcome. Moving the gate
  is not recommended."

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHY THIS RECOMMENDATION IS DEFENSIBLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1. This was a randomized experiment — causal inference is valid.
     Unlike observational studies, we do not need to worry about
     confounding variables. The only difference between groups was
     the gate position.

  2. gate_30 won on all three measured outcomes — engagement,
     1-day retention, and 7-day retention. No metric favored
     gate_40. The direction of evidence is completely consistent.

  3. The only significant result (retention_7) favors gate_30,
     and the confidence interval for the difference does not
     include zero — the direction is reliable, not a fluke.

  4. There is no upside to moving the gate. gate_40 does not
     improve any outcome. A change with no measurable benefit
     and a measurable cost (lower 7-day retention) should not
     be implemented.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  1. NOVELTY EFFECT NOT ACCOUNTED FOR
     Players in the treatment group (gate_40) encountered a
     different experience than they may have expected. Some
     behavior change could reflect novelty — players exploring
     further simply because something changed — rather than a
     genuine response to the gate position. This effect typically
     fades over time and is not captured in a snapshot experiment.

  2. NO PLAYER SEGMENTATION
     The analysis treats all players as one group. In reality,
     new players and experienced players may respond very
     differently to gate position. A player hitting a gate at
     level 30 on their first session is a different situation
     from a returning player hitting it. Segmented analysis
     could reveal effects hidden in the aggregate.

  3. SNAPSHOT, NOT LONGITUDINAL
     Retention was measured at day 1 and day 7 only. We do not
     know whether gate position affects retention at day 14,
     day 30, or beyond. The long-term trajectory of player
     behavior under each condition is unknown.

  4. NO MONETIZATION DATA
     The dataset contains no revenue or purchase data. It is
     possible that gate_40 produces higher monetization even
     with slightly lower retention — players who reach level 40
     may be more engaged spenders. This analysis cannot address
     that question.

  5. EFFECT SIZE IS SMALL
     Even the significant finding (retention_7) has a negligible
     effect size (Cramer's V = 0.0105). While the recommendation
     is clear given consistent direction of evidence, no single
     outcome showed a large or even medium effect. The business
     impact of keeping gate_30 is real but modest.

  6. EXTERNAL VALIDITY
     This experiment reflects one game, one time period, and one
     player population. Results may not generalize to different
     games, different gate levels (e.g., gate_20 vs gate_50),
     or future player cohorts with different demographics or
     gaming habits.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INTERVIEW HEADLINE (one sentence)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  "In a randomized experiment of 90,000 players, I found that
  moving Cookie Cats' gate from level 30 to level 40 produced
  no improvement in engagement or short-term retention, and
  caused a statistically significant reduction in 7-day retention
  — leading to a clear recommendation to keep the gate at level 30,
  supported by a reusable statistical testing framework I built
  to automate test selection and effect size calculation."
"""

print(recommendation)

# --- Save ---
rec_path = os.path.join(project_root, 'data', 'phase4_recommendation.txt')
with open(rec_path, 'w', encoding='utf-8') as f:
    f.write(recommendation)
print(f"Recommendation saved to: {rec_path}")


╔══════════════════════════════════════════════════════════════════════╗
║              FINAL BUSINESS RECOMMENDATION & LIMITATIONS             ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FINAL RECOMMENDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Do NOT move the gate to level 40.

  Keep the gate at level 30 (gate_30).

  Full recommendation statement:
  "Based on a properly randomized experiment of 90,188 players,
  moving the gate from level 30 to level 40 shows no improvement
  in player engagement or 1-day retention, and produces a small
  but statistically significant reduction in 7-day retention
  (19.02% vs 18.20%, p=0.0016). While the effect is modest in
  percentage-point terms, 7-day retention is the most meaningful
  long-term health metric available in this dataset, and gate_30
  outperforms gate_40 on every measured outcome. Moving the g